# 움직임 시험 6 — 영상 모델 움직임 + LoRA 덧칠: 시험 3(Wan VACE, 참조 있음) 결과를 SDXL+LoRA+윤곽 조절로 장마다 덧칠 (Kaggle·Colab T4, 학습 없음)
시험 3: VACE는 배경·움직임이 장 사이에 일관됐지만 그림체가 일반 만화 폭발(우리 LoRA 없음). 시험 4: SDXL+LoRA는 화풍이 붙지만 장마다 배경이 바뀜.
→ VACE 장을 시작점으로 낮은 강도(배경·구도 유지) + 그 장의 윤곽 조절로 LoRA 화풍만 입힌다.
- **s0.45**: 강도 0.45 + 윤곽 0.6 (구도 거의 유지, 화풍 약하게)
- **s0.65**: 강도 0.65 + 윤곽 0.7 (화풍 강하게, 흔들림 위험)
- 같은 LoRA(3차 3바퀴)·시드(1234). 입력: Drive `vace_in.zip`(시험 3 num·clip 참조 있음 결과 + 블로킹).
- 설정(Kaggle): **GPU T4 x2**, **Internet On** → Save & Run All. 약 30–50분. Colab: T4 GPU → 모두 실행.
- 끝나면 `motion6_result.zip`(나란히 영상 `motion6.mp4` + 장별 그림 + 깜빡임 수치)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 0) 어디서 돌리나: Kaggle이면 /kaggle/working, Colab이면 /content/out 에 결과를 쓴다
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
# 1) 패키지와 자료: 기본 모델(HF), LoRA 3차 3바퀴·시험 3 결과(Drive)
# 버전 고정: Colab·Kaggle 기본 파이썬 3.13 + 새 transformers에서 diffusers 0.31이 깨졌다(FLAX_WEIGHTS_NAME, 2026-10-26). 서버 py3.13에서 아래 조합 import 확인
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "peft==0.17.1" "accelerate>=1.0" opencv-python-headless
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/lora.safetensors "https://drive.usercontent.google.com/download?id=1ht_UqE8JkJTUX-eO5lNIqvhyFRcprUWJ&export=download&confirm=t"
os.system('curl -sSL -o /tmp/vace_in.zip "https://drive.usercontent.google.com/download?id=1GIMBb1T3lGqkZHvhNT6qhx2PpvuV-gsH&export=download&confirm=t"')
os.system("cd /tmp && unzip -q -o /tmp/vace_in.zip")
for name in ("num", "clip"):
    print(name, len(glob.glob(f"/tmp/m3/{name}/ref/*.png")), "장 VACE,", len(glob.glob(f"/tmp/m3/{name}/blocking/*.png")), "장 블로킹")
!ls -la /tmp/ilxl.safetensors /tmp/lora.safetensors


In [ ]:
# 2) 파이프라인: SDXL ControlNet 그림→그림 + LoRA 합치기 (윤곽 조절 xinsir/controlnet-canny-sdxl-1.0)
import torch
from diffusers import StableDiffusionXLControlNetImg2ImgPipeline, ControlNetModel, EulerAncestralDiscreteScheduler
from PIL import Image
cn = ControlNetModel.from_pretrained("xinsir/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_single_file("/tmp/ilxl.safetensors", controlnet=cn, torch_dtype=torch.float16)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("/tmp/lora.safetensors", adapter_name="fx")
pipe.fuse_lora(lora_scale=1.0); pipe.unload_lora_weights()
pipe.to("cuda")
pipe.vae.to(torch.float32)
print(torch.cuda.memory_allocated() / 1e9, "GB")


In [ ]:
# 3) VACE 장마다 LoRA 덧칠
import numpy as np, json, cv2
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, city, night, wide shot"
N = "lowres, worst quality, text, watermark"
STEPS, CFG, SEED = 28, 6.0, 1234
W, H = 1344, 768
def edges(b):
    e = cv2.Canny(cv2.cvtColor(np.asarray(b), cv2.COLOR_RGB2GRAY), 50, 150)
    return Image.fromarray(np.stack([e] * 3, -1))
def run(init, ctrl, s, cs):
    g = torch.Generator("cuda").manual_seed(SEED)
    with torch.inference_mode():
        return pipe(prompt=P, negative_prompt=N, image=init, control_image=ctrl, strength=s, controlnet_conditioning_scale=cs,
                    num_inference_steps=STEPS, guidance_scale=CFG, generator=g, width=W, height=H).images[0]
def diff(a, b):
    return float(np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32)).mean())
log = {}
for name in ("num", "clip"):
    vace = [Image.open(p).convert("RGB").resize((W, H), Image.LANCZOS) for p in sorted(glob.glob(f"/tmp/m3/{name}/ref/*.png"))]
    blk = [Image.open(p).convert("RGB").resize((W, H)) for p in sorted(glob.glob(f"/tmp/m3/{name}/blocking/*.png"))]
    T = min(len(vace), len(blk)); vace, blk = vace[:T], blk[:T]
    for tag, s, cs in (("s045", 0.45, 0.6), ("s065", 0.65, 0.7)):
        ims = [run(v, edges(v), s, cs) for v in vace]
        d = f"{OUT}/m6/{name}/{tag}"; os.makedirs(d, exist_ok=True)
        for i, im in enumerate(ims):
            im.save(f"{d}/f{i:03d}.png")
        log[f"{name} {tag}"] = round(float(np.mean([abs(diff(ims[i + 1], ims[i]) - diff(blk[i + 1], blk[i])) for i in range(T - 1)])), 2)
        print(name, tag, T, "장, 깜빡임", log[f"{name} {tag}"])
    log[f"{name} vace"] = round(float(np.mean([abs(diff(vace[i + 1], vace[i]) - diff(blk[i + 1], blk[i])) for i in range(T - 1)])), 2)
    for sub, ims in (("blocking", blk), ("vace", vace)):
        d0 = f"{OUT}/m6/{name}/{sub}"; os.makedirs(d0, exist_ok=True)
        for i, im in enumerate(ims):
            im.save(f"{d0}/f{i:03d}.png")
print(log)
json.dump(log, open(f"{OUT}/motion6_flicker.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# 4) 나란히 영상: 위 줄 num, 아래 줄 clip / 칸 = 블로킹 | VACE | 덧칠 0.45 | 덧칠 0.65. 24fps, 3번 반복
rows = {n: {k: sorted(glob.glob(f"{OUT}/m6/{n}/{k}/*.png")) for k in ("blocking", "vace", "s045", "s065")} for n in ("num", "clip")}
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (336, 192)); cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2); return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter(f"{OUT}/motion6.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 24, (1344, 384))
for _ in range(3):
    for t in range(T):
        vw.write(np.vstack([np.hstack([rd(v[t % len(v)], f"{n} {k}") for k, v in rows[n].items()]) for n in rows]))
vw.release()
!cd {OUT} && zip -q -r motion6_result.zip m6 motion6.mp4 motion6_flicker.json && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭에서 받음)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion6_result.zip")
